#use analy environment

In [1]:
#write config files for all models to SFT them on hellaswag, piqa, and safety datasets


###what this script does:
    #start with model's config file for metamathqa
    #change dataset
    #change output dir -- from llamafactory_out to scratch node
    #change run name
    #(don't need to change model since it's the same as the starting config)

In [1]:
import shutil
from itertools import product

In [2]:
##args -- write new config files with this setup

#arguments that change -- re-run notebook for each
sft_dataset = 'safetymixed300'



#arguments that do not change
model_dict = {
    'llama-0.5B-10BT': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 1.5, 3.0, 10.0],
    'llama-1B-20BT': [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 1.5, 3.0, 10.0],
    'llama-4B-80BT': [0.1, 1.0],
    'olmo-1B-30BT': [0.1, 0.3, 0.6, 1.0],
    'olmo-1B-210BT': [0.1, 0.3, 1.0],
}

In [4]:
n_newconfig_files_created = 0


for model_size, wd_lst in model_dict.items():
    print(f"Creating config file for: {model_size}")

    for wd_pt in wd_lst:
        print(f"   wd={wd_pt}")

        # make a copy of a metamathqa config, then rewrite fields for the target dataset
        if model_size in ['llama-0.5B-10BT', 'llama-1B-20BT', 'llama-4B-80BT']:
            model_folder_name = f'{model_size}-weightdecay{wd_pt}-seed42'
            og_config_file_path = f'config_hub/custom_configs/ft_metamathqa/{model_size}-weightdecay{wd_pt}-seed42-metamathqa.yaml'
            new_config_file_path = f'config_hub/custom_configs/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}.yaml'
        elif model_size in ['olmo-1B-30BT', 'olmo-1B-210BT']:
            model_folder_name = f'{model_size}-weightdecay{wd_pt}'
            og_config_file_path = f'config_hub/custom_configs/ft_metamathqa/{model_size}-weightdecay{wd_pt}-metamathqa.yaml'
            new_config_file_path = f'config_hub/custom_configs/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-{sft_dataset}.yaml'
        else:
            raise ValueError(f"Unknown model_size format: {model_size}")

        shutil.copyfile(og_config_file_path, new_config_file_path)

        # make edits to the new config file
        with open(new_config_file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # changes these variables in config file:
        # output_dir
        # sft_dataset
        # run_name

        # change output_dir
        old_output_dir = f'llamafactory_out/{model_folder_name}'
        new_output_dir = f'/n/netscratch/doshi-velez_lab/Everyone/models/ft_new_tasks/{model_folder_name}'
        text = text.replace(old_output_dir, new_output_dir)

        # change appearances of sft_dataset -- effectively changing sft_dataset and run_name
        text = text.replace('metamathqa', sft_dataset)

        # write new config file
        with open(new_config_file_path, "w", encoding="utf-8") as f:
            f.write(text)

        n_newconfig_files_created += 1

print(n_newconfig_files_created)
print("Complete!")


Creating config file for: llama-0.5B-10BT
   wd=0.0001
   wd=0.001
   wd=0.01
   wd=0.1
   wd=0.5
   wd=1.0
   wd=1.5
   wd=3.0
   wd=10.0
Creating config file for: llama-1B-20BT
   wd=0.0001
   wd=0.001
   wd=0.01
   wd=0.1
   wd=0.5
   wd=1.0
   wd=1.5
   wd=3.0
   wd=10.0
Creating config file for: llama-4B-80BT
   wd=0.1
   wd=1.0
Creating config file for: olmo-1B-30BT
   wd=0.1
   wd=0.3
   wd=0.6
   wd=1.0
Creating config file for: olmo-1B-210BT
   wd=0.1
   wd=0.3
   wd=1.0
27
Complete!
